# Validation de la segmentation de SAM2 et construction de la matrice de features pour les analyses



*   En entrée ce code prend un fichier excel qui a deux sheets, le premier est le tableau binaire des intéraction entre souches, le deuxième étant le tableau des intéractions provenant du labo
*   L'idée est de faire une validation du tableau de SAM2



In [1]:
# Packages nécessaires
import pandas as pd
from itertools import product

In [2]:
FICHIER = "../data/sortie/variable_halo_VF.xlsx"
SHEET_HALO = "halo_image"
SHEET_VERIF = "verif"

df_halo = pd.read_excel(FICHIER, sheet_name=SHEET_HALO, dtype=object)
df_halo["cible"] = df_halo["cible"].astype(str).str.strip()
df_halo["productrice"] = df_halo["productrice"].astype(str).str.strip()

cibles_halo = set(df_halo["cible"].unique())
prods_halo  = set(df_halo["productrice"].unique())

df_verif = pd.read_excel(FICHIER, sheet_name=SHEET_VERIF, dtype=object)
row_label = df_verif.columns[0]                 # ex: 'Unnamed: 0'
verif = df_verif.set_index(row_label)

verif.index = verif.index.astype(str).str.strip()
verif.columns = verif.columns.astype(str).str.strip()

cibles_verif = set(verif.index.tolist())
prods_verif  = set(verif.columns.tolist())

# Comparaisons entre halo_image et verif
extra_cibles_in_halo = sorted(cibles_halo - cibles_verif)
extra_prods_in_halo  = sorted(prods_halo  - prods_verif)
missing_cibles_in_halo = sorted(cibles_verif - cibles_halo)
missing_prods_in_halo  = sorted(prods_verif  - prods_halo)

print("\nComparaison entre halo_image et verif")
print("---------------------------------------")

print(f"\nCibles présentes dans halo_image mais absentes de verif ({len(extra_cibles_in_halo)}): {extra_cibles_in_halo}")
print(f"Productrices présentes dans halo_image mais absentes de verif ({len(extra_prods_in_halo)}): {extra_prods_in_halo}")

print(f"\nCibles présentes dans verif mais absentes dans halo_image ({len(missing_cibles_in_halo)}): {missing_cibles_in_halo}")
print(f"Productrices présentes dans verif mais absentes dans halo_image ({len(missing_prods_in_halo)}): {missing_prods_in_halo}\n")

# Comptage des observations dans halo_image
nb_halo = len(df_halo)

print(f"Nombre d'observations dans 'halo_image' : {nb_halo}")
print(f"Nb cibles dans verif (lignes)    : {len(cibles_verif)}")
print(f"Nb productrices dans verif (cols): {len(prods_verif)}")
print(f"Nb cibles dans halo_image        : {len(cibles_halo)}")
print(f"Nb productrices dans halo_image  : {len(prods_halo)}\n")

# Couples attendus par verif mais absents dans halo_image
# Carte complète des couples d'après verif (tous croisements lignes x colonnes)
full_pairs = pd.DataFrame(
    [(c, p) for c, p in product(sorted(cibles_verif), sorted(prods_verif))],
    columns=["cible", "productrice"]
)

# Marquer ceux présents dans halo_image
pairs_halo = df_halo[["cible", "productrice"]].drop_duplicates()
pairs_halo["present_halo"] = True

full_pairs = full_pairs.merge(pairs_halo, how="left", on=["cible", "productrice"])
missing_pairs = full_pairs[full_pairs["present_halo"].isna()].drop(columns=["present_halo"])

print(f"Nb de couples possibles selon verif : {len(full_pairs)}")
print(f"Nb de couples présents dans halo_image : {len(pairs_halo)}")
print(f"Nb de couples manquants dans halo_image (attendus par verif) : {len(missing_pairs)}\n")

# Aperçu
print("Aperçu des 20 premiers couples manquants :")
print(missing_pairs.head(69).to_string(index=False))


Comparaison entre halo_image et verif
---------------------------------------

Cibles présentes dans halo_image mais absentes de verif (0): []
Productrices présentes dans halo_image mais absentes de verif (0): []

Cibles présentes dans verif mais absentes dans halo_image (1): ['1008']
Productrices présentes dans verif mais absentes dans halo_image (0): []

Nombre d'observations dans 'halo_image' : 1452
Nb cibles dans verif (lignes)    : 39
Nb productrices dans verif (cols): 39
Nb cibles dans halo_image        : 38
Nb productrices dans halo_image  : 39

Nb de couples possibles selon verif : 1521
Nb de couples présents dans halo_image : 1452
Nb de couples manquants dans halo_image (attendus par verif) : 69

Aperçu des 20 premiers couples manquants :
cible productrice
 1008        1001
 1008        1003
 1008        1006
 1008        1008
 1008        1009
 1008        1011
 1008        1012
 1008        1014
 1008        1015
 1008        1017
 1008        1019
 1008        1020
 1008  

In [3]:
import pandas as pd

# On supprime les observations sur un essai dans vérif
# qui correspond a la valeur  "O", on ajoute aussi dans le dataframe final
# les observations présentes dans le fichier de vérification et absente
# dans notre tableau.

if "couple" not in df_halo.columns:
    df_halo["couple"] = df_halo["productrice"] + "_" + df_halo["cible"]

# --- Lecture de la matrice verif ---
df_verif = pd.read_excel(FICHIER, sheet_name=SHEET_VERIF, dtype=object)
first_col = df_verif.columns[0]
verif = df_verif.set_index(first_col)
verif.index = verif.index.astype(str).str.strip()
verif.columns = verif.columns.astype(str).str.strip()

# --- Conversion des cellules de verif ---
def verif_cell_to_binary(val):
    if pd.isna(val):
        return 0
    s = str(val).strip().upper()
    if s == "X":
        return 1
    elif s == "O":
        return None
    else:
        return 0

# --- Extraction de la valeur attendue depuis verif ---
def expected_from_verif(row):
    cible = row["cible"]
    prod = row["productrice"]
    try:
        val = verif.loc[cible, prod]
        return verif_cell_to_binary(val)
    except KeyError:
        return None

df_halo["expected"] = df_halo.apply(expected_from_verif, axis=1)

# --- Suppression des lignes marquées 'O' ---
mask_O = df_halo["expected"].isna()
df_suppr = df_halo[mask_O].copy()
df = df_halo[~mask_O].copy()
print(f"Lignes supprimées (valeurs 'O' dans verif) : {len(df_suppr)}")

# --- Correction des halos selon verif ---
mod_counts = {"0->1": 0, "1->0": 0, "NA->1": 0, "NA->0": 0, "1->1": 0, "0->0": 0}
comparisons = []

def correct_halo(row):
    h = row["halo"]
    e = row["expected"]
    if pd.isna(e):
        return h
    e_val = int(e)
    if pd.isna(h):
        mod_counts[f"NA->{e_val}"] += 1
        comparisons.append(("NA", str(e_val)))
        return e_val
    h_val = int(h)
    if h_val != e_val:
        mod_counts[f"{h_val}->{e_val}"] += 1
        comparisons.append((str(h_val), str(e_val)))
        return e_val
    comparisons.append((str(h_val), str(e_val)))
    mod_counts[f"{h_val}->{h_val}"] += 1
    return h_val

df["halo_corrige"] = df.apply(correct_halo, axis=1).astype("Int64")

# --- Ajout des couples manquants depuis verif ---
existing_pairs = set(zip(df["cible"], df["productrice"]))
all_pairs = {(cible, prod) for cible in verif.index for prod in verif.columns}
missing_pairs = sorted(all_pairs - existing_pairs)

new_rows = []
for cible, prod in missing_pairs:
    val = verif.loc[cible, prod]
    halo_val = verif_cell_to_binary(val)
    if halo_val is not None:  # ignore les 'O'
        new_rows.append({
            "cible": cible,
            "productrice": prod,
            "couple": f"{prod}_{cible}",
            "halo": halo_val
        })

df_add = pd.DataFrame(new_rows)
print(f"Couples ajoutés depuis verif : {len(df_add)}")

# --- Fusion finale ---
df_main = (
    df.drop(columns=["expected", "halo"], errors="ignore")
      .rename(columns={"halo_corrige": "halo"})
      .reset_index(drop=True)
)
df_add = df_add.reset_index(drop=True)
out = pd.concat([df_main, df_add], ignore_index=True)

# --- Sauvegarde finale (toutes feuilles en un seul passage) ---
summary = pd.DataFrame(comparisons, columns=["halo_image", "verif"])
table_summary = summary.value_counts().reset_index(name="count")
table_summary = table_summary.pivot(index="halo_image", columns="verif", values="count").fillna(0).astype(int)


# --- Rapport global ---
total_modifs = sum(v for k, v in mod_counts.items() if "->" in k)
print("\nRAPPORT FINAL")
print(f"Total initial : {len(df_halo)}")
print(f"Supprimées ('O') : {len(df_suppr)}")
print(f"Corrigées : {total_modifs}")
for k, v in mod_counts.items():
    print(f"   {k} : {v}")
print(f"Ajouts : {len(df_add)}")
print(f"Total final : {len(out)}")

print("\nTABLEAU DE SYNTHÈSE DES COMPARAISONS")
print(table_summary)

print("\nScript exécuté avec succès.")

Lignes supprimées (valeurs 'O' dans verif) : 216
Couples ajoutés depuis verif : 62

RAPPORT FINAL
Total initial : 1452
Supprimées ('O') : 216
Corrigées : 1236
   0->1 : 173
   1->0 : 2
   NA->1 : 1
   NA->0 : 61
   1->1 : 77
   0->0 : 922
Ajouts : 62
Total final : 1298

TABLEAU DE SYNTHÈSE DES COMPARAISONS
verif         0    1
halo_image          
0           922  173
1             2   77
NA           61    1

Script exécuté avec succès.


In [4]:
# --- Fonction de conversion cohérente avec le script principal ---
def verif_to_binary(val):
    if pd.isna(val):
        return 0
    s = str(val).strip().upper()
    if s == "X":
        return 1
    elif s == "O":
        return None  # on ignore les 'O'
    else:
        return 0

# --- Couples existants et manquants ---
pairs_existing = set(zip(df_halo["cible"], df_halo["productrice"]))
all_pairs = {(c, p) for c in verif.index for p in verif.columns}
missing_pairs = sorted(all_pairs - pairs_existing)

# --- Construction des nouvelles lignes ---
new_rows = []
report = {}

for cible, prod in missing_pairs:
    val = verif.loc[cible, prod] if (cible in verif.index and prod in verif.columns) else pd.NA
    halo_val = verif_to_binary(val)
    if halo_val is not None:  # on ignore les 'O'
        new_rows.append({
            "cible": cible,
            "productrice": prod,
            "couple": f"{prod}_{cible}",
            "halo": halo_val
        })
        report[cible] = report.get(cible, 0) + 1

# --- Rapport ---
print(f"Nombre total de couples ajoutés : {len(new_rows)}\n")
for cible, n in sorted(report.items(), key=lambda x: x[0]):
    print(f"- Souche cible {cible} : {n} couples ajoutés")

Nombre total de couples ajoutés : 62

- Souche cible 1008 : 35 couples ajoutés
- Souche cible 993 : 9 couples ajoutés
- Souche cible 994 : 18 couples ajoutés


In [5]:
# --- Vérification post-correction ---

# On vérifie que pour chaque couple (cible, productrice) où halo = 1
# dans le DataFrame 'out' (halo corrigé), la matrice 'verif' contient bien un 'X'.

ok = err = 0

# Normalisation des chaînes, au cas où
out["cible"] = out["cible"].astype(str).str.strip()
out["productrice"] = out["productrice"].astype(str).str.strip()

for _, r in out[out["halo"] == 1].iterrows():
    cible = r["cible"]
    prod = r["productrice"]
    try:
        val = verif.loc[cible, prod]
        if isinstance(val, str) and val.strip().upper() == "X":
            ok += 1
        else:
            err += 1
    except KeyError:
        err += 1

print("\nVÉRIFICATION POST-CORRECTION")
print(f"Halos = 1 avec 'X' dans verif : {ok}")
print(f"Halos = 1 sans 'X' dans verif : {err}")


VÉRIFICATION POST-CORRECTION
Halos = 1 avec 'X' dans verif : 253
Halos = 1 sans 'X' dans verif : 0


In [6]:
# --- Vérification : cellules vides dans verif ---

# Objectif : repérer les cellules vides dans verif
# et vérifier que, pour ces couples (cible, productrice),
# le halo correspondant dans 'out' vaut bien 0 (ou n'existe pas).

ok = err = 0
erreurs = []

# Normalisation au cas où
out["cible"] = out["cible"].astype(str).str.strip()
out["productrice"] = out["productrice"].astype(str).str.strip()

for cible in verif.index:
    for prod in verif.columns:
        val = verif.loc[cible, prod]
        # On cible les cellules vides
        if pd.isna(val) or str(val).strip() == "":
            # On cherche le couple correspondant dans out
            match = out[(out["cible"] == cible) & (out["productrice"] == prod)]
            if not match.empty:
                halo_val = match["halo"].astype(str).str.strip().unique()
                if "0" in halo_val:
                    ok += 1  # correct : halo=0
                else:
                    err += 1
                    erreurs.append((prod, cible, halo_val))
            else:
                ok += 1  # pas de couple => considéré comme halo=0 implicite

print("\nVÉRIFICATION POST-CORRECTION (cellules vides dans verif)")
print(f"Cellules vides avec halo = 0 : {ok}")
print(f"Cellules vides avec halo ≠ 0 : {err}")

if err > 0:
    print("\nExemples de divergences (jusqu’à 10) :")
    for e in erreurs[:10]:
        print(f"Productrice={e[0]}, Cible={e[1]}, Halo(s)={e[2]}")


VÉRIFICATION POST-CORRECTION (cellules vides dans verif)
Cellules vides avec halo = 0 : 1045
Cellules vides avec halo ≠ 0 : 0


## Création du dataset complet d’interactions dans la cellule qui suit

Ce script utilise le tableau des halos corrigés déjà chargé en mémoire
(DataFrame `out`) et le fichier 'jointures.xlsx' (les profils LC_MS) pour obtenir un
Dataframe complet prêt pour analyse.

➡ Fonctionnement :
   1. Jointure sur la colonne 'productrice' → ajoute les infos de la
      souche productrice (suffixe '_P')
   2. Jointure sur la colonne 'cible' → ajoute les infos de la
      souche cible (suffixe '_C')
   
➡ Résultat :
   Un tableau où chaque ligne correspond à une interaction :
       productrice | cible | halo | ... + infos productrice (_P) + infos cible (_C)

In [7]:
FICHIER_X = "../data/Features_souches/jointures.xlsx"

# Lecture du fichier des métadonnées
df_X = pd.read_excel(FICHIER_X, sheet_name="X_Design", dtype=object)

# Normalisation des colonnes identifiants
df_X["souche"] = df_X["Souche"].astype(str).str.strip()
out["productrice"] = out["productrice"].astype(str).str.strip()
out["cible"] = out["cible"].astype(str).str.strip()

# --- Jointure pour productrice ---
df_prod = out.merge(
    df_X, left_on="productrice", right_on="souche", how="left", suffixes=("", "_drop")
)
df_prod = df_prod.drop(columns=["souche"], errors="ignore")

# Renommer les colonnes explicatives avec suffixe _P
rename_prod = {col: f"{col}_P" for col in df_X.columns if col != "souche"}
df_prod = df_prod.rename(columns=rename_prod)

# --- Jointure pour cible ---
df_complet = df_prod.merge(
    df_X, left_on="cible", right_on="souche", how="left", suffixes=("", "_drop")
)
df_complet = df_complet.drop(columns=["souche"], errors="ignore")

# Renommer les colonnes explicatives avec suffixe _C
rename_cible = {col: f"{col}_C" for col in df_X.columns if col != "souche"}
df_complet = df_complet.rename(columns=rename_cible)

In [8]:
# --- Suppression des auto-inhibitions (cible = productrice) ---
nb_total = len(df_complet)
df_filtre = df_complet[df_complet["cible"] != df_complet["productrice"]].copy()
nb_suppr = nb_total - len(df_filtre)

# --- Sauvegarde du dataset filtré ---
output = "../data/sortie/dataset_complet_sans_auto_inhibition.csv"
df_filtre.to_csv(output, index=False)

# --- Rapport final ---
print("\nFILTRAGE DES AUTO-INHIBITIONS")
print(f"Fichier filtré sauvegardé sous : {output}")
print(f"Lignes supprimées (cible = productrice) : {nb_suppr}")
print(f"Total initial : {nb_total} → Total final : {len(df_filtre)}")


FILTRAGE DES AUTO-INHIBITIONS
Fichier filtré sauvegardé sous : ../data/sortie/dataset_complet_sans_auto_inhibition.csv
Lignes supprimées (cible = productrice) : 39
Total initial : 1298 → Total final : 1259
